<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 생성 모델 · 07. DDPM

## Denoising Diffusion Probabilistic Models

- **원 논문:** [Denoising Diffusion Probabilistic Models](https://papers.nips.cc/paper_files/paper/2020/hash/4c5bcfec8584af0d967f1ab10179ca4b-Abstract.html)
- **저자 / 발표:** Jonathan Ho, Ajay Jain, and Pieter Abbeel · NeurIPS (2020)
- **난이도 / 예상 시간:** 고급 · 약 70분
- **선수 지식:** Gaussian Markov chain, variance schedule, neural noise prediction
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 8×8 로컬 영상에서 closed-form forward noising, epsilon objective, reverse sampling을 구현한다.

**축소하거나 생략한 부분:** 논문은 긴 diffusion chain과 U-Net으로 CIFAR-10 등을 생성한다. 여기서는 T=20과 MLP denoiser로 Algorithm 1–2를 보존한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (2) and Eq. (4) | forward process | q(x_t|x_0)를 누적 alpha_bar로 한 번에 표본화한다. |
| §3.2, Eq. (11) | epsilon parameterization | reverse mean을 predicted noise로 표현한다. |
| §3.4, Eq. (14), Algorithm 1 and Algorithm 2 | simple objective and algorithms | 무작위 t의 noise MSE로 학습하고 역순 sampling한다. |

## 핵심 재현

        Eq. (4) $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$로 임의 시점을
        한 번에 만들고 Eq. (14)의 $\|\epsilon-\epsilon_\theta(x_t,t)\|^2$를 최소화합니다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(371)
np.random.seed(371)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,\qquad \mathcal L=\mathbb E\lVert\epsilon-\epsilon_\theta(x_t,t)\rVert_2^2
$$

- **기호와 역할:** $t$는 timestep, $\bar\alpha_t$는 누적 signal 비율, $\epsilon_\theta$는 주입한 Gaussian noise predictor입니다.
- **shape 계약:** `x0/xt/noise [N, 64], t [N] → concat [N, 65] → predicted noise [N, 64]`
- **논문 위치:** `§2, Eq. (2) and Eq. (4)`의 **forward process**
- **코드 Task:** forward noising, epsilon loss, reverse mean과 Algorithm 2 sampling을 구현합니다.
- **학습·평가 흐름:** 무작위 t/noise → q_sample → noise MSE update → t=T-1..0 reverse sample
- **원문 대비 한계:** 논문은 긴 diffusion chain과 U-Net으로 CIFAR-10 등을 생성한다. 여기서는 T=20과 MLP denoiser로 Algorithm 1–2를 보존한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

In [ ]:
class DiffusionProcessLab(nn.Module):
    """DDPM의 핵심 학습·평가 경로. 입출력 계약: x0/xt/noise [N, 64], t [N] → concat [N, 65] →
    predicted noise [N, 64]."""

    def __init__(self, noise_predictor, optimizer, beta_schedule):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        super().__init__()
        self.noise_predictor = noise_predictor
        self.optimizer = optimizer
        self.register_buffer("beta_schedule", beta_schedule)
        alpha_schedule = 1.0 - beta_schedule
        self.register_buffer("alpha_schedule", alpha_schedule)
        self.register_buffer("alpha_bar", torch.cumprod(alpha_schedule, dim=0))

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"timesteps": len(self.beta_schedule), "data_dim": 64}

    def forward(self, clean, timestep, noise):
        """논문 순전파를 계산한다. shape 계약: x0/xt/noise [N, 64], t [N] → concat [N, 65] →
        predicted noise [N, 64]."""
        signal = self.alpha_bar[timestep].sqrt().reshape(-1, 1)
        noise_scale = (1.0 - self.alpha_bar[timestep]).sqrt().reshape(-1, 1)
        noisy = signal * clean + noise_scale * noise
        predicted_noise = self.predict_noise(noisy, timestep)
        return (noisy, predicted_noise)

    def predict_noise(self, noisy, timestep):
        """논문 알고리즘의 공개 보조 연산을 수행한다."""
        time_feature = timestep[:, None] / (len(self.beta_schedule) - 1)
        return self.noise_predictor(torch.cat([noisy, time_feature], dim=1))

    def loss(self, clean, timestep, noise):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        _, predicted_noise = self.forward(clean, timestep, noise)
        return (predicted_noise - noise).square().mean()

    def update(self, clean, timestep, noise):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(clean, timestep, noise)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        return float(objective.detach())

    def reverse_step(self, noisy, step, generator):
        """논문 알고리즘의 공개 보조 연산을 수행한다."""
        timestep = torch.full(
            (len(noisy),), step, dtype=torch.long, device=noisy.device
        )
        predicted_noise = self.predict_noise(noisy, timestep)
        mean = noisy - self.beta_schedule[step] * predicted_noise / torch.sqrt(
            1.0 - self.alpha_bar[step]
        )
        mean = mean / torch.sqrt(self.alpha_schedule[step])
        if step == 0:
            return mean
        innovation = randn_device(noisy.shape, generator=generator)
        return mean + self.beta_schedule[step].sqrt() * innovation

    def sample(self, batch_size, generator):
        """논문 알고리즘의 공개 보조 연산을 수행한다."""
        state = randn_device(
            (batch_size, self.config()["data_dim"]), generator=generator
        )
        trajectory = [state.detach().clone()]
        with torch.no_grad():
            for step in reversed(range(len(self.beta_schedule))):
                state = self.reverse_step(state, step, generator)
                trajectory.append(state.detach().clone())
        return (state, torch.stack(trajectory))

    def evaluate(self, clean, timestep, noise):
        """no_grad 경로에서 논문과 연결된 정량 지표 dict를 반환한다."""
        with torch.no_grad():
            noisy, predicted_noise = self.forward(clean, timestep, noise)
            mse = (predicted_noise - noise).square().mean()
            sample, trajectory = self.sample(
                min(8, len(clean)), torch.Generator().manual_seed(4071)
            )
        return {
            "noise_mse": float(mse),
            "noisy_shape": tuple(noisy.shape),
            "sample_shape": tuple(sample.shape),
            "trajectory_steps": len(trajectory),
            "sample_abs_mean": float(sample.abs().mean()),
        }

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§2, Eq. (2) and Eq. (4)` — **forward process**
- **구현 이유:** q(x_t|x_0)를 누적 alpha_bar로 한 번에 표본화한다.
- **읽을 코드:** 아래 `implementation` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
T = 20
beta = torch.linspace(1e-4, 0.08, T, device=DEVICE)
alpha = 1 - beta
alpha_bar = torch.cumprod(alpha, 0)


def extract(v, t, x):
    return v[t].reshape(-1, *([1] * (x.ndim - 1)))


def q_sample(x0, t, noise):
    return (
        extract(alpha_bar.sqrt(), t, x0) * x0
        + extract((1 - alpha_bar).sqrt(), t, x0) * noise
    )


probe = images_01[:4] * 2 - 1
zero_t = torch.zeros(4, dtype=torch.long, device=DEVICE)
eps = torch.zeros_like(probe)
assert torch.allclose(q_sample(probe, zero_t, eps), alpha_bar[0].sqrt() * probe)

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3.4, Eq. (14), Algorithm 1 and Algorithm 2` — **simple objective and algorithms**
- **구현 이유:** 무작위 t의 noise MSE로 학습하고 역순 sampling한다.
- **읽을 코드:** 아래 `training` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
denoiser = nn.Sequential(nn.Linear(65, 96), nn.SiLU(), nn.Linear(96, 64)).to(DEVICE)
opt = torch.optim.Adam(denoiser.parameters(), lr=0.004)
x0 = images_01.flatten(1) * 2 - 1
rng = torch.Generator().manual_seed(372)
history = []
with torch.no_grad():
    t0 = randint_device(T, (len(x0),), generator=rng)
    n0 = randn_device(x0.shape, generator=rng)
    initial = float(
        F.mse_loss(
            denoiser(torch.cat([q_sample(x0, t0, n0), t0[:, None] / (T - 1)], 1)),
            n0,
        )
    )

portfolio_model = DiffusionProcessLab(denoiser, opt, beta)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in denoiser.parameters()]
)
for _ in range(180):
    timestep = randint_device(T, (len(x0),), generator=rng)
    noise = randn_device(x0.shape, generator=rng)
    history.append(portfolio_model.update(x0, timestep, noise))
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in denoiser.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert history[-1] < initial and parameter_delta > 0.0
assert np.isfinite(history).all()

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§3.4, Eq. (14), Algorithm 1 and Algorithm 2` — **simple objective and algorithms**
        - **구현 이유:** 무작위 t의 noise MSE로 학습하고 역순 sampling한다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
evaluation_timestep = torch.zeros(len(x0), dtype=torch.long, device=DEVICE)
evaluation_noise = randn_device(x0.shape, generator=torch.Generator().manual_seed(373))
portfolio_metrics = portfolio_model.evaluate(x0, evaluation_timestep, evaluation_noise)
assert portfolio_metrics

sample_rng = torch.Generator().manual_seed(373)
x = randn_device((8, 64), generator=sample_rng)
with torch.no_grad():
    for step in reversed(range(T)):
        t = torch.full((len(x),), step, dtype=torch.long, device=DEVICE)
        pred = denoiser(torch.cat([x, t[:, None] / (T - 1)], 1))
        mean = (x - beta[step] / torch.sqrt(1 - alpha_bar[step]) * pred) / torch.sqrt(
            alpha[step]
        )
        x = (
            mean
            if step == 0
            else mean + beta[step].sqrt() * randn_device(x.shape, generator=sample_rng)
        )
assert x.shape == (8, 64) and torch.isfinite(x).all()
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
axes[0, 0].plot(history)
axes[0, 0].set_title("noise MSE")
axes[1, 0].axis("off")
for ax, img in zip(axes[:, 1:].ravel(), x.detach().cpu()):
    ax.imshow(img.reshape(8, 8).clamp(-1, 1), cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")
fig.tight_layout()
plt.show()

t에 대한 균등 Monte Carlo 평균은 모든 timestep loss 합의 불편추정량입니다. Eq. (4) 덕분에 앞선 모든 step을 순차 실행하지 않아도 x_t를 만들 수 있습니다.

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§2, Eq. (2) and Eq. (4)`의 **forward process**
- **핵심 식:**

  $$
  x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,\qquad \mathcal L=\mathbb E\lVert\epsilon-\epsilon_\theta(x_t,t)\rVert_2^2
  $$

- **입출력 shape:** `x0/xt/noise [N, 64], t [N] → concat [N, 65] → predicted noise [N, 64]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `DiffusionProcessLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. q(x_t|x_0)를 누적 alpha_bar로 한 번에 표본화한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_model.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_model).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 긴 diffusion chain과 U-Net으로 CIFAR-10 등을 생성한다. 여기서는 T=20과 MLP denoiser로 Algorithm 1–2를 보존한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.